# Vesuvius Surface Detection - Topology-Aware Training

**Key improvements over baseline:**
1. **TopologyAwareLoss**: Combo (Dice+CE) + clDice for topology preservation
2. **SwinUNETR / UNETR**: Transformer architectures alongside UNet
3. **3D augmentations**: Flips, rotations, intensity shifts
4. **Mixed precision**: AMP for faster training
5. **Cosine annealing + warmup**: Learning rate schedule

---

## 0. Setup

In [ ]:
!pip install -q monai tifffile imagecodecs scikit-image

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from glob import glob
import tifffile
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from monai.networks.nets import UNet, SwinUNETR, UNETR

import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects

# Settings
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 1. Configuration

In [ ]:
class CFG:
    # Paths (Kaggle)
    root_dir = '/kaggle/input/vesuvius-challenge-surface-detection'
    train_image_dir = f'{root_dir}/train_images'
    train_label_dir = f'{root_dir}/train_labels'
    output_dir = '/kaggle/working'
    
    # Model
    model_name = 'unet'  # 'unet', 'swinunetr', 'unetr'
    in_channels = 1
    out_channels = 3  # background, foreground, ignore
    
    # Training
    epochs = 80
    batch_size = 1
    lr = 1e-4
    weight_decay = 1e-5
    patch_size = (160, 160, 160)
    num_patches_per_volume = 4
    val_split = 0.15
    
    # Loss
    loss_name = 'topology'  # 'combo', 'cldice', 'topology'
    combo_weight = 0.7  # Weight for combo loss in topology loss
    cldice_weight = 0.3  # Weight for clDice in topology loss
    
    # Scheduler
    warmup_epochs = 5
    
    # AMP
    use_amp = True
    
    # Workers
    num_workers = 2
    
    seed = 42

print(f'Model: {CFG.model_name}')
print(f'Loss: {CFG.loss_name}')
print(f'Patch size: {CFG.patch_size}')
print(f'Epochs: {CFG.epochs}')

## 2. Loss Functions

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5, ignore_index=2):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, pred, target):
        pred = F.softmax(pred, dim=1)
        valid_mask = (target != self.ignore_index).float()
        num_classes = pred.shape[1]
        target_oh = F.one_hot(
            target.clamp(0, num_classes - 1), num_classes
        ).permute(0, 4, 1, 2, 3).float()
        
        pred = pred * valid_mask.unsqueeze(1)
        target_oh = target_oh * valid_mask.unsqueeze(1)
        
        dice_scores = []
        for c in range(1, num_classes):
            intersection = (pred[:, c] * target_oh[:, c]).sum()
            union = pred[:, c].sum() + target_oh[:, c].sum()
            dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)
        
        return 1.0 - torch.stack(dice_scores).mean()


class SoftSkeletonize3D(nn.Module):
    """Differentiable 3D skeletonization via iterative erosion."""
    def __init__(self, num_iterations=10):
        super().__init__()
        self.num_iterations = num_iterations
        kernel = torch.ones(1, 1, 3, 3, 3) / 27.0
        self.register_buffer('kernel', kernel)

    def forward(self, x):
        skeleton = torch.zeros_like(x)
        for i in range(self.num_iterations):
            eroded = F.conv3d(x, self.kernel, padding=1)
            eroded = (eroded > 0.5).float()
            diff = x - eroded
            skeleton = torch.max(skeleton, diff * (i + 1) / self.num_iterations)
            x = eroded
            if x.sum() == 0:
                break
        return skeleton


class ClDiceLoss(nn.Module):
    """Centerline Dice Loss for topology preservation."""
    def __init__(self, smooth=1e-5, ignore_index=2, alpha=0.5, num_iters=10):
        super().__init__()
        self.smooth = smooth
        self.ignore_index = ignore_index
        self.alpha = alpha
        self.skeletonize = SoftSkeletonize3D(num_iters)

    def forward(self, pred, target):
        pred_prob = F.softmax(pred, dim=1)[:, 1:2]
        target_bin = (target == 1).float().unsqueeze(1)
        valid = (target != self.ignore_index).float().unsqueeze(1)
        
        pred_prob = pred_prob * valid
        target_bin = target_bin * valid
        
        # Regular dice
        inter = (pred_prob * target_bin).sum()
        dice = (2 * inter + self.smooth) / (pred_prob.sum() + target_bin.sum() + self.smooth)
        
        # Skeleton-based topology metrics
        pred_skel = self.skeletonize(pred_prob)
        target_skel = self.skeletonize(target_bin)
        
        tprec = (pred_skel * target_bin).sum() / (pred_skel.sum() + self.smooth)
        tsens = (target_skel * pred_prob).sum() / (target_skel.sum() + self.smooth)
        cl_dice = (2 * tprec * tsens + self.smooth) / (tprec + tsens + self.smooth)
        
        combined = self.alpha * cl_dice + (1 - self.alpha) * dice
        return 1.0 - combined


class TopologyAwareLoss(nn.Module):
    """Combined Combo + clDice loss."""
    def __init__(self, combo_w=0.7, cldice_w=0.3, ignore_index=2):
        super().__init__()
        self.combo_w = combo_w
        self.cldice_w = cldice_w
        self.dice_loss = DiceLoss(ignore_index=ignore_index)
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.cldice_loss = ClDiceLoss(ignore_index=ignore_index)

    def forward(self, pred, target):
        combo = 0.5 * self.dice_loss(pred, target) + 0.5 * self.ce_loss(pred, target)
        cldice = self.cldice_loss(pred, target)
        return self.combo_w * combo + self.cldice_w * cldice


def get_loss(name, **kwargs):
    if name == 'combo':
        dice = DiceLoss()
        ce = nn.CrossEntropyLoss(ignore_index=2)
        return lambda p, t: 0.5 * dice(p, t) + 0.5 * ce(p, t)
    elif name == 'cldice':
        return ClDiceLoss(**kwargs)
    elif name == 'topology':
        return TopologyAwareLoss(**kwargs)
    else:
        raise ValueError(f'Unknown loss: {name}')

print('Loss functions defined.')

## 3. Dataset

In [ ]:
class VesuviusDataset(Dataset):
    def __init__(self, image_paths, label_paths, patch_size=(160,160,160),
                 num_patches=4, augment=False):
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.patch_size = patch_size
        self.num_patches = num_patches
        self.augment = augment
    
    def __len__(self):
        return len(self.image_paths) * self.num_patches
    
    def _load(self, idx):
        vol_idx = idx // self.num_patches
        img = tifffile.imread(self.image_paths[vol_idx]).astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        mask = tifffile.imread(self.label_paths[vol_idx]).astype(np.int64)
        return img, mask
    
    def _random_patch(self, img, mask):
        d, h, w = img.shape
        pd, ph, pw = self.patch_size
        
        d0 = np.random.randint(0, max(1, d - pd + 1))
        h0 = np.random.randint(0, max(1, h - ph + 1))
        w0 = np.random.randint(0, max(1, w - pw + 1))
        
        ip = img[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        mp = mask[d0:d0+pd, h0:h0+ph, w0:w0+pw]
        
        # Pad if needed
        if ip.shape != self.patch_size:
            pad = [(0, max(0, s - ip.shape[i])) for i, s in enumerate(self.patch_size)]
            ip = np.pad(ip, pad, mode='constant', constant_values=0)
            mp = np.pad(mp, pad, mode='constant', constant_values=2)
            ip = ip[:pd, :ph, :pw]
            mp = mp[:pd, :ph, :pw]
        
        return ip, mp
    
    def _augment(self, img, mask):
        # Random flips
        for ax in [0, 1, 2]:
            if np.random.random() > 0.5:
                img = np.flip(img, axis=ax).copy()
                mask = np.flip(mask, axis=ax).copy()
        
        # Random rotation in HW plane
        if np.random.random() > 0.5:
            k = np.random.randint(1, 4)
            img = np.rot90(img, k=k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k=k, axes=(1, 2)).copy()
        
        # Intensity augmentation
        if np.random.random() > 0.7:
            img = img + np.random.uniform(-0.1, 0.1)
            img = np.clip(img, 0, 1)
        
        if np.random.random() > 0.7:
            factor = np.random.uniform(0.8, 1.2)
            mean = img.mean()
            img = (img - mean) * factor + mean
            img = np.clip(img, 0, 1)
        
        if np.random.random() > 0.8:
            img = img + np.random.normal(0, 0.02, img.shape).astype(np.float32)
            img = np.clip(img, 0, 1)
        
        return img, mask
    
    def __getitem__(self, idx):
        img, mask = self._load(idx)
        img, mask = self._random_patch(img, mask)
        
        if self.augment:
            img, mask = self._augment(img, mask)
        
        img = torch.from_numpy(img).unsqueeze(0).float()  # (1, D, H, W)
        mask = torch.from_numpy(mask).long()
        
        return img, mask

print('Dataset defined.')

## 4. Data Preparation

In [ ]:
# Get all training files
train_images = sorted(glob(f'{CFG.train_image_dir}/*.tif'))
train_labels = sorted(glob(f'{CFG.train_label_dir}/*.tif'))

# Verify alignment
assert len(train_images) == len(train_labels), \
    f'Mismatch: {len(train_images)} images, {len(train_labels)} labels'

for img_p, lbl_p in zip(train_images[:3], train_labels[:3]):
    assert Path(img_p).stem == Path(lbl_p).stem, f'Mismatch: {img_p} vs {lbl_p}'

print(f'Total training volumes: {len(train_images)}')

# Train/val split
np.random.seed(CFG.seed)
indices = np.random.permutation(len(train_images))
split = int(len(indices) * (1 - CFG.val_split))

train_idx = indices[:split]
val_idx = indices[split:]

train_img_paths = [train_images[i] for i in train_idx]
train_lbl_paths = [train_labels[i] for i in train_idx]
val_img_paths = [train_images[i] for i in val_idx]
val_lbl_paths = [train_labels[i] for i in val_idx]

print(f'Train: {len(train_img_paths)}, Val: {len(val_img_paths)}')

# Create datasets
train_dataset = VesuviusDataset(
    train_img_paths, train_lbl_paths,
    patch_size=CFG.patch_size,
    num_patches=CFG.num_patches_per_volume,
    augment=True
)
val_dataset = VesuviusDataset(
    val_img_paths, val_lbl_paths,
    patch_size=CFG.patch_size,
    num_patches=2,
    augment=False
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.batch_size,
    shuffle=True, num_workers=CFG.num_workers,
    pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.batch_size,
    shuffle=False, num_workers=CFG.num_workers,
    pin_memory=True
)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## 5. Model

In [ ]:
def get_model(name, img_size=(160, 160, 160)):
    if name == 'unet':
        return UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=3,
            channels=(32, 64, 128, 256, 512),
            strides=(2, 2, 2, 2),
            num_res_units=2,
        )
    elif name == 'swinunetr':
        return SwinUNETR(
            img_size=img_size,
            in_channels=1,
            out_channels=3,
            feature_size=48,
            depths=(2, 2, 2, 2),
            num_heads=(3, 6, 12, 24),
            spatial_dims=3,
        )
    elif name == 'unetr':
        return UNETR(
            in_channels=1,
            out_channels=3,
            img_size=img_size,
            feature_size=16,
            hidden_size=768,
            mlp_dim=3072,
            num_heads=12,
            spatial_dims=3,
        )
    else:
        raise ValueError(f'Unknown model: {name}')

model = get_model(CFG.model_name, CFG.patch_size).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG.model_name}')
print(f'Parameters: {n_params:,} ({n_params/1e6:.2f}M)')

## 6. Training Loop

In [ ]:
# Loss, optimizer, scheduler
criterion = get_loss(CFG.loss_name)

optimizer = optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG.epochs - CFG.warmup_epochs,
    eta_min=1e-7
)

warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    total_iters=CFG.warmup_epochs
)

scaler = GradScaler() if CFG.use_amp else None

print(f'Loss: {CFG.loss_name}')
print(f'Optimizer: AdamW (lr={CFG.lr})')
print(f'Scheduler: CosineAnnealing + {CFG.warmup_epochs}-epoch warmup')
print(f'AMP: {CFG.use_amp}')

In [ ]:
def compute_dice(pred, target, ignore_index=2):
    """Compute foreground Dice score."""
    with torch.no_grad():
        pred_cls = pred.argmax(dim=1)
        valid = target != ignore_index
        pred_fg = (pred_cls == 1) & valid
        target_fg = (target == 1) & valid
        inter = (pred_fg & target_fg).sum().float()
        return (2 * inter / (pred_fg.sum() + target_fg.sum() + 1e-8)).item()


best_dice = 0
history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': [], 'lr': []}

for epoch in range(CFG.epochs):
    # ---- TRAIN ----
    model.train()
    train_loss, train_dice, n_train = 0, 0, 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG.epochs} [Train]')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        dice = compute_dice(outputs, labels)
        train_loss += loss.item()
        train_dice += dice
        n_train += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}', dice=f'{dice:.4f}')
    
    # ---- VALIDATE ----
    model.eval()
    val_loss, val_dice, n_val = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{CFG.epochs} [Val]'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            dice = compute_dice(outputs, labels)
            
            val_loss += loss.item()
            val_dice += dice
            n_val += 1
    
    # Update scheduler
    if epoch < CFG.warmup_epochs:
        warmup_scheduler.step()
    else:
        scheduler.step()
    
    # Log
    avg_train_loss = train_loss / n_train
    avg_val_loss = val_loss / n_val
    avg_train_dice = train_dice / n_train
    avg_val_dice = val_dice / n_val
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['lr'].append(current_lr)
    
    print(f'\nEpoch {epoch+1}: Train Loss={avg_train_loss:.4f} Dice={avg_train_dice:.4f} | '
          f'Val Loss={avg_val_loss:.4f} Dice={avg_val_dice:.4f} | LR={current_lr:.6f}')
    
    # Save best
    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_dice': best_dice,
        }, f'{CFG.output_dir}/best_model.pth')
        print(f'  >>> New best Dice: {best_dice:.4f}')
    
    # Save periodic
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
        }, f'{CFG.output_dir}/epoch_{epoch+1}.pth')

print(f'\nTraining complete! Best val Dice: {best_dice:.4f}')

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(history['train_dice'], label='Train')
axes[1].plot(history['val_dice'], label='Val')
axes[1].set_title('Dice')
axes[1].legend()
axes[1].set_xlabel('Epoch')

axes[2].plot(history['lr'])
axes[2].set_title('Learning Rate')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(f'{CFG.output_dir}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Inference with TTA + Post-Processing

In [ ]:
# Load best model
checkpoint = torch.load(f'{CFG.output_dir}/best_model.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {checkpoint["epoch"]+1} (Dice: {checkpoint["best_dice"]:.4f})')

In [ ]:
# Sliding Window Inference
def sliding_window_inference(model, volume, roi_size=(160,160,160), overlap=0.5):
    """Run sliding window inference with Gaussian weighting."""
    d, h, w = volume.shape
    pd, ph, pw = roi_size
    
    step_d = int(pd * (1 - overlap))
    step_h = int(ph * (1 - overlap))
    step_w = int(pw * (1 - overlap))
    
    # Gaussian weights
    sigma = min(pd, ph, pw) / 4
    z = torch.arange(pd).float() - pd/2
    y = torch.arange(ph).float() - ph/2
    x = torch.arange(pw).float() - pw/2
    gw = (torch.exp(-z**2/(2*sigma**2))[:, None, None] *
          torch.exp(-y**2/(2*sigma**2))[None, :, None] *
          torch.exp(-x**2/(2*sigma**2))[None, None, :]).numpy()
    
    output = None
    weight_sum = np.zeros((d, h, w), dtype=np.float32)
    
    positions = []
    for zi in range(0, max(1, d - pd + 1), step_d):
        for yi in range(0, max(1, h - ph + 1), step_h):
            for xi in range(0, max(1, w - pw + 1), step_w):
                positions.append((zi, yi, xi))
    # Edge positions
    if d > pd: positions.append((d-pd, 0, 0))
    if h > ph: positions.append((0, h-ph, 0))
    if w > pw: positions.append((0, 0, w-pw))
    if d > pd and h > ph and w > pw:
        positions.append((d-pd, h-ph, w-pw))
    positions = list(set(positions))
    
    with torch.no_grad():
        for zi, yi, xi in positions:
            patch = volume[zi:zi+pd, yi:yi+ph, xi:xi+pw]
            if patch.shape != roi_size:
                pads = [(0, max(0, s - patch.shape[i])) for i, s in enumerate(roi_size)]
                patch = np.pad(patch, pads, mode='constant')
            
            inp = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device)
            pred = model(inp).cpu().numpy()[0]  # (C, D, H, W)
            
            if output is None:
                nc = pred.shape[0]
                output = np.zeros((nc, d, h, w), dtype=np.float32)
            
            vd = min(pd, d - zi)
            vh = min(ph, h - yi)
            vw = min(pw, w - xi)
            
            for c in range(nc):
                output[c, zi:zi+vd, yi:yi+vh, xi:xi+vw] += \
                    pred[c, :vd, :vh, :vw] * gw[:vd, :vh, :vw]
            weight_sum[zi:zi+vd, yi:yi+vh, xi:xi+vw] += gw[:vd, :vh, :vw]
    
    weight_sum = np.maximum(weight_sum, 1e-8)
    for c in range(output.shape[0]):
        output[c] /= weight_sum
    
    return output  # (C, D, H, W)


def predict_with_tta(model, volume, roi_size=(160,160,160), overlap=0.5):
    """TTA: 8 views (original + 3 flips + 3 rotations)."""
    logits_list = []
    
    # Original
    logits_list.append(sliding_window_inference(model, volume, roi_size, overlap))
    
    # Flips
    for axis in [0, 1, 2]:
        vol_f = np.flip(volume, axis=axis).copy()
        pred = sliding_window_inference(model, vol_f, roi_size, overlap)
        pred = np.flip(pred, axis=axis + 1).copy()  # +1 for class dim
        logits_list.append(pred)
    
    # Rotations in HW
    for k in [1, 2, 3]:
        vol_r = np.rot90(volume, k=k, axes=(1, 2)).copy()
        pred = sliding_window_inference(model, vol_r, roi_size, overlap)
        pred = np.rot90(pred, k=-k, axes=(2, 3)).copy()
        logits_list.append(pred)
    
    return np.mean(logits_list, axis=0)  # (C, D, H, W)


print('Inference functions defined.')

In [ ]:
# Post-processing
def build_anisotropic_struct(z_radius, xy_radius):
    z, r = z_radius, xy_radius
    if z == 0 and r == 0: return None
    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy*dy + dx*dx <= r*r:
                    struct[cz+dz, cy+dy, cx+dx] = True
    return struct


def topo_postprocess(class_map, t_low=0.30, t_high=0.80, z_radius=3, xy_radius=2, dust_min=100):
    """Hysteresis + closing + dust removal."""
    strong = class_map >= t_high
    weak = class_map >= t_low
    if not strong.any():
        return np.zeros_like(class_map, dtype=np.uint8)
    
    struct = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct)
    if not mask.any():
        return np.zeros_like(class_map, dtype=np.uint8)
    
    struct_close = build_anisotropic_struct(z_radius, xy_radius)
    if struct_close is not None:
        mask = ndi.binary_closing(mask, structure=struct_close)
    
    if dust_min > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min)
    
    return mask.astype(np.uint8)

print('Post-processing functions defined.')

## 9. Create Submission

In [ ]:
import zipfile

test_df = pd.read_csv(f'{CFG.root_dir}/test.csv')
test_dir = f'{CFG.root_dir}/test_images'
zip_path = f'{CFG.output_dir}/submission.zip'
mask_dir = f'{CFG.output_dir}/submission_masks'
os.makedirs(mask_dir, exist_ok=True)

# PP parameters
PP = dict(t_low=0.30, t_high=0.80, z_radius=3, xy_radius=2, dust_min=100)

print(f'Test samples: {len(test_df)}')
print(f'Post-processing: {PP}')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        image_id = row['id']
        path = f'{test_dir}/{image_id}.tif'
        
        # Load and normalize
        vol = tifffile.imread(path).astype(np.float32)
        vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-8)
        
        print(f'\nProcessing {image_id} - shape: {vol.shape}')
        
        # Predict with TTA
        logits = predict_with_tta(model, vol, roi_size=CFG.patch_size, overlap=0.4)
        class_map = logits.argmax(axis=0).astype(np.float32)  # (D, H, W)
        
        # Post-process
        mask = topo_postprocess(class_map, **PP)
        
        fg_pct = mask.sum() / mask.size * 100
        print(f'  Foreground: {mask.sum():,} voxels ({fg_pct:.2f}%)')
        
        # Save
        out_path = f'{mask_dir}/{image_id}.tif'
        tifffile.imwrite(out_path, mask)
        zf.write(out_path, arcname=f'{image_id}.tif')
        os.remove(out_path)

print(f'\nSubmission saved to: {zip_path}')

In [ ]:
# Verify submission
with zipfile.ZipFile(zip_path, 'r') as zf:
    files = zf.namelist()
    print(f'Files in submission: {len(files)}')
    for f in files:
        info = zf.getinfo(f)
        print(f'  {f}: {info.compress_size/1024:.1f} KB')
    
expected = set(test_df['id'].astype(str).tolist())
actual = set([f.replace('.tif', '') for f in files])
assert expected == actual, f'Mismatch! Missing: {expected - actual}'
print('Submission verified.')